# Bài 11 · Xử lý dữ liệu phi cấu trúc bằng LLM

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Viện TTNT, UET-VNU**

Notebook bám theo slide Bài 11. Chạy tuần tự từ trên xuống.

> 💡 **Trước khi sửa:** File → *Save a copy in Drive* để lưu bản của riêng bạn.

**Mục tiêu buổi học** — sau buổi này, bạn:

1. Giải thích được vì sao dữ liệu văn bản tự do phải **đổi dạng thành bảng** trước khi phân tích.
2. Xây được **baseline không-LLM** (từ khoá/từ điển) cho bài trích xuất thông tin.
3. Gọi được **Gemini API** từ Python với **structured output** (schema + enum), có rate limit + cache.
4. **Đánh giá** chất lượng LLM so với baseline trên bộ nhãn gán tay, và ước tính **chi phí** bằng token.

## 0. Chuẩn bị

### Lấy API key (miễn phí, không cần thẻ)

1. Vào [Google AI Studio](https://aistudio.google.com) → đăng nhập Google → **Get API key** → tạo key.
2. Trong Colab: bấm biểu tượng **🔑 Secrets** ở cột trái → **Add new secret** → Name: `GEMINI_API_KEY`, Value: dán key → bật **Notebook access**.

### Không có key thì sao?

Vẫn học được: notebook có sẵn **kết quả LLM đã cache** cho dữ liệu demo (`DEMO_MODE`).
Mọi cell đều chạy — chỉ khác là kết quả LLM đọc từ cache thay vì gọi API thật.
Đây cũng chính là kỹ thuật cache mà bài tập lớn yêu cầu (cờ `--skip-llm`).

In [ ]:
%pip install -q google-genai pydantic

import json, re, time, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Đọc API key: Colab Secrets -> biến môi trường -> không có (DEMO_MODE)
API_KEY = None
try:
    from google.colab import userdata          # chỉ có trên Colab
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    API_KEY = os.environ.get("GEMINI_API_KEY")

HAS_KEY = bool(API_KEY)
MODEL = "gemini-3.1-flash-lite"   # quota free cao; thử "gemini-3.5-flash" với văn bản khó

if HAS_KEY:
    from google import genai
    client = genai.Client(api_key=API_KEY)
    print(f"✅ Có API key — sẽ gọi Gemini thật (model: {MODEL})")
else:
    print("ℹ️ Không có API key — DEMO_MODE: dùng kết quả LLM đã cache. Notebook vẫn chạy đủ.")

## 1. Dữ liệu demo: 24 review chỗ ở

Trích 24 review "kiểu Inside Airbnb" (rút gọn, nhiều ngôn ngữ) — đủ nhỏ để ta **đọc được bằng mắt
toàn bộ**, điều không thể làm với 690.000 review thật. Cột `comments` là thứ ta xử lý trong buổi này.

In [ ]:
REVIEWS = [
    (1,  "Great location, five minutes from the metro. The host was lovely!"),
    (2,  "The flat was not clean at all, dust everywhere. Location was perfect though."),
    (3,  "Súper cómodo y muy limpio. El anfitrión respondió al instante."),
    (4,  "Ruidoso hasta las 3am, imposible dormir. No lo recomiendo."),
    (5,  "Apartamento bem localizado, mas o wifi caiu o tempo todo."),
    (6,  "Nothing special, but it was cheap and did the job."),
    (7,  "L'appartement est charmant mais la rue est très bruyante la nuit."),
    (8,  "Amazing value for money, spotless bathroom, super quiet neighborhood."),
    (9,  "Todo perfecto! Volveremos seguro."),
    (10, "The pictures are misleading — the room is tiny and smells of smoke."),
    (11, "Host cancelled last minute, we were stranded. Avoid!"),
    (12, "Muito limpo, cama confortável, anfitriã atenciosa. Recomendo!"),
    (13, "Decent place. A bit far from the center but the bus stop is close."),
    (14, "Perfect for a weekend getaway."),
    (15, "El barrio se siente inseguro de noche; el depto está bien."),
    (16, "Ottima posizione, appartamento pulito. Torneremo!"),
    (17, "Quiet street, comfy bed, kind host — everything was great."),
    (18, "We found hair in the sheets and the kitchen was greasy."),
    (19, "So loud! The bar downstairs plays music till 2am."),
    (20, "Un poco caro para lo que ofrece, pero la ubicación es inmejorable."),
    (21, "The check-in instructions were confusing; host took hours to reply."),
    (22, "Cozy studio, fast wifi, walkable to everything."),
    (23, "👍👍👍"),
    (24, ""),
]
df = pd.DataFrame(REVIEWS, columns=["review_id", "comments"]).set_index("review_id")
print(f"{len(df)} review; độ dài trung bình {df.comments.str.len().mean():.0f} ký tự")
df.head(6)

Câu hỏi ta muốn trả lời (thu nhỏ từ câu hỏi thật của bài tập lớn):

> **Khách chê điều gì nhiều nhất? Cảm xúc chung ra sao? Review viết bằng những ngôn ngữ nào?**

Muốn `groupby` được thì trước hết phải **tạo ra các cột** `sentiment`, `aspects`, `language` — chúng
chưa tồn tại.

## 2. Baseline không cần LLM

Nguyên tắc làm việc: **baseline trước, công cụ đắt tiền sau**. Baseline cho ta thước đo — nếu LLM
không hơn baseline thì tội gì trả tiền (và chờ đợi)?

In [ ]:
# 2a. Khía cạnh (aspect) bằng từ khoá — mỗi khía cạnh một pattern regex
# (bắt đầu thực tế: soạn bằng tiếng Anh — ngôn ngữ mình đọc được)
ASPECT_KEYWORDS = {
    "location":    r"location|metro|center|walkable",
    "cleanliness": r"clean|dust|dirty|hair|greasy|smell",
    "host":        r"host|check-in",
    "value":       r"value|price|cheap",
    "noise":       r"noisy|noise|loud|quiet",
    "amenities":   r"wifi|bed|kitchen|bathroom",
}

def baseline_aspects(text: str) -> list[str]:
    return [a for a, pat in ASPECT_KEYWORDS.items()
            if re.search(pat, text, flags=re.I)]

df["base_aspects"] = df["comments"].map(baseline_aspects)
df[["comments", "base_aspects"]].head(8)

In [ ]:
# 2b. Cảm xúc bằng từ điển: đếm từ tích cực trừ từ tiêu cực (cũng tiếng Anh)
POS_WORDS = r"great|perfect|amazing|excellent|lovely|comfy|cozy|spotless|kind|fast|quiet"
NEG_WORDS = r"not |dirty|noisy|loud|avoid|cancelled|misleading|smell|hair|greasy|confusing|stranded|tiny"

def baseline_sentiment(text: str) -> str:
    pos = len(re.findall(POS_WORDS, text, flags=re.I))
    neg = len(re.findall(NEG_WORDS, text, flags=re.I))
    return "positive" if pos > neg else "negative" if neg > pos else "mixed"

df["base_sent"] = df["comments"].map(baseline_sentiment)
df["base_sent"].value_counts()

In [ ]:
# 2c. Ngôn ngữ bằng "stopword đặc trưng" — heuristic 10 dòng
LANG_HINTS = {
    "en": r"\b(the|was|and|but|from)\b",
    "es": r"\b(el|la|muy|pero|hasta|todo)\b",
    "pt": r"\b(o|muito|mas|tempo|bem)\b",
    "fr": r"\b(le|la|est|très|mais)\b",
    "it": r"\b(ottima|molto|appartamento)\b",
}

def baseline_lang(text: str) -> str:
    counts = {lang: len(re.findall(pat, text, flags=re.I))
              for lang, pat in LANG_HINTS.items()}
    best = max(counts, key=counts.get)
    return best if counts[best] > 0 else "und"   # und = undetermined

df["base_lang"] = df["comments"].map(baseline_lang)
df[["comments", "base_sent", "base_lang"]].tail(6)

Nhìn kết quả trên đã thấy vài "mùi" quen thuộc:

- Review 2 *"not clean at all"* — dính từ khoá `clean`, baseline không hiểu phủ định.
- Các review tiếng Tây Ban Nha / Bồ Đào Nha / Ý gần như **tàng hình** với bộ từ khoá tiếng Anh —
  muốn phủ thêm một ngôn ngữ là phải soạn lại từ điển cho ngôn ngữ đó.
- Review 23 (toàn emoji) và 24 (rỗng) — baseline bó tay, trả `und`/`mixed`.

Đừng vội kết luận — **cảm giác chưa phải bằng chứng**. Mục 5 sẽ đo đàng hoàng.

## 3. Gọi LLM từ Python

Cách dùng LLM trong pipeline khác hẳn cách dùng chat: code gửi **request**, nhận **response**, lặp
được hàng nghìn lần. Thử lần gọi đầu tiên (cell này chỉ chạy khi có key):

In [ ]:
if HAS_KEY:
    interaction = client.interactions.create(
        model=MODEL,
        input="Chào lớp Lập trình xử lý dữ liệu bằng đúng một câu tiếng Việt.",
    )
    print(interaction.output_text)
else:
    print("(DEMO_MODE — bỏ qua lần gọi thử. Kết quả thật sẽ kiểu: 'Chào cả lớp…')")

### Vấn đề của output tự do

Hỏi "review này khen gì chê gì" sẽ nhận về **văn xuôi** — mỗi lần một kiểu, không parse nổi thành cột.
Giải pháp: **structured output** — khai một *mẫu trả lời* (schema), mô hình bị ràng buộc điền đúng mẫu.

## 4. Structured output: bắt LLM điền form

### 4a. Khai schema bằng Pydantic

`Literal` đóng vai enum: nhãn ngoài danh sách là **không hợp lệ** — hàng rào đầu tiên chống bịa nhãn.

In [ ]:
from pydantic import BaseModel
from typing import Literal

Aspect = Literal["location", "cleanliness", "host", "value", "noise", "amenities"]

class ReviewInfo(BaseModel):
    sentiment: Literal["positive", "mixed", "negative"]
    aspects_positive: list[Aspect]   # khía cạnh được KHEN
    aspects_negative: list[Aspect]   # khía cạnh bị CHÊ
    language: str                    # mã ISO 639-1: "en", "es", ... ("und" nếu không rõ)

PROMPT_TEMPLATE = """Bạn là công cụ trích xuất thông tin từ review chỗ ở.
Phân tích review sau và điền đúng schema. Chỉ gán khía cạnh THỰC SỰ được nhắc đến;
review không có nội dung thì trả sentiment "mixed" và các danh sách rỗng.

Review: {text}"""

print(json.dumps(ReviewInfo.model_json_schema(), indent=2)[:400], "…")

### 4b. Cache — đã gọi thì không gọi lại

Ta gói lời gọi API vào một hàm có **cache**: kết quả từng review được lưu lại theo `review_id`.
Chạy lại notebook không tốn thêm request nào; không có key thì đọc cache có sẵn — chính là cơ chế
`--skip-llm` mà bài tập lớn yêu cầu.

*(Cell dưới dài vì chứa cache đã tính sẵn cho 24 review — cứ chạy, không cần đọc từng dòng.)*

In [ ]:
# Kết quả LLM đã cache cho 24 review demo (sinh bằng gemini-3.1-flash-lite khi soạn bài, 07/2026)
LLM_CACHE = json.loads("""{
 "1": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"location\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 172,
  "out_tok": 38
 },
 "2": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [\\"location\\"], \\"aspects_negative\\": [\\"cleanliness\\"], \\"language\\": \\"en\\"}",
  "in_tok": 181,
  "out_tok": 44
 },
 "3": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"cleanliness\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"es\\"}",
  "in_tok": 175,
  "out_tok": 40
 },
 "4": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"noise\\"], \\"language\\": \\"es\\"}",
  "in_tok": 174,
  "out_tok": 34
 },
 "5": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [\\"location\\"], \\"aspects_negative\\": [\\"amenities\\"], \\"language\\": \\"pt\\"}",
  "in_tok": 176,
  "out_tok": 42
 },
 "6": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [\\"value\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 170,
  "out_tok": 33
 },
 "7": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"noise\\"], \\"language\\": \\"fr\\"}",
  "in_tok": 178,
  "out_tok": 36
 },
 "8": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"value\\", \\"cleanliness\\", \\"noise\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 179,
  "out_tok": 46
 },
 "9": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"pt\\"}",
  "in_tok": 165,
  "out_tok": 28
 },
 "10": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"cleanliness\\"], \\"language\\": \\"en\\"}",
  "in_tok": 177,
  "out_tok": 35
 },
 "11": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"host\\"], \\"language\\": \\"en\\"}",
  "in_tok": 173,
  "out_tok": 33
 },
 "12": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"cleanliness\\", \\"amenities\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"pt\\"}",
  "in_tok": 180,
  "out_tok": 44
 },
 "13": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"location\\"], \\"language\\": \\"en\\"}",
  "in_tok": 175,
  "out_tok": 36
 },
 "14": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"location\\", \\"cleanliness\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 166,
  "out_tok": 37
 },
 "15": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"es\\"}",
  "in_tok": 176,
  "out_tok": 30
 },
 "16": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"location\\", \\"cleanliness\\"], \\"aspects_negative\\": [], \\"language\\": \\"it\\"}",
  "in_tok": 171,
  "out_tok": 38
 },
 "17": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"noise\\", \\"amenities\\", \\"host\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 177,
  "out_tok": 42
 },
 "18": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"cleanliness\\"], \\"language\\": \\"en\\"}",
  "in_tok": 174,
  "out_tok": 34
 },
 "19": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"noise\\"], \\"language\\": \\"en\\"}",
  "in_tok": 172,
  "out_tok": 33
 },
 "20": {
  "output": "{\\"sentiment\\": \\"mixed\\", \\"aspects_positive\\": [\\"location\\"], \\"aspects_negative\\": [\\"value\\"], \\"language\\": \\"es\\"}",
  "in_tok": 179,
  "out_tok": 43
 },
 "21": {
  "output": "{\\"sentiment\\": \\"negative\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [\\"host\\"], \\"language\\": \\"en\\"}",
  "in_tok": 176,
  "out_tok": 39
 },
 "22": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [\\"amenities\\", \\"location\\"], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 173,
  "out_tok": 36
 },
 "23": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"und\\"}",
  "in_tok": 163,
  "out_tok": 26
 },
 "24": {
  "output": "{\\"sentiment\\": \\"positive\\", \\"aspects_positive\\": [], \\"aspects_negative\\": [], \\"language\\": \\"en\\"}",
  "in_tok": 160,
  "out_tok": 24
 }
}""")
print(f"Cache có sẵn {len(LLM_CACHE)} kết quả")

In [ ]:
def call_llm_raw(text: str):
    """Gọi Gemini thật, trả về (json_str, in_tok, out_tok)."""
    interaction = client.interactions.create(
        model=MODEL,
        input=PROMPT_TEMPLATE.format(text=text),
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": ReviewInfo.model_json_schema(),
        },
    )
    u = interaction.usage
    return interaction.output_text, u.total_input_tokens, u.total_output_tokens

def extract(review_id: int, text: str, sleep_s: float = 4.0) -> ReviewInfo | None:
    """Trích xuất 1 review, ưu tiên cache; retry 1 lần khi lỗi tạm thời."""
    key = str(review_id)
    if key not in LLM_CACHE:                       # chỉ gọi API khi chưa có cache
        if not HAS_KEY:
            return None                            # DEMO_MODE mà thiếu cache -> chịu
        for attempt in range(2):
            try:
                out, tin, tout = call_llm_raw(text)
                LLM_CACHE[key] = {"output": out, "in_tok": tin, "out_tok": tout}
                time.sleep(sleep_s)                # tôn trọng rate limit (RPM) free tier
                break
            except Exception as e:
                print(f"  review {review_id} lỗi lần {attempt+1}: {e}")
                time.sleep(20)
        else:
            return None
    try:
        return ReviewInfo.model_validate_json(LLM_CACHE[key]["output"])
    except Exception as e:                          # schema không khớp -> loại, không cho vào bảng
        print(f"  review {review_id}: output không hợp lệ ({e})")
        return None

In [ ]:
# Chạy trích xuất cho toàn bộ 24 review (demo: đọc cache nên chạy tức thì)
results = {rid: extract(rid, text) for rid, text in df["comments"].items()}
ok = {rid: r for rid, r in results.items() if r is not None}
print(f"Trích xuất hợp lệ: {len(ok)}/{len(df)}")

### 4c. Về lại thế giới quen: DataFrame

Kết quả trích xuất giờ ghép lại thành **bảng nhãn** — từ đây mọi thứ là pandas thuần.

In [ ]:
labels = pd.DataFrame({rid: r.model_dump() for rid, r in ok.items()}).T
labels.index.name = "review_id"
full = df.join(labels, how="left")
full[["comments", "sentiment", "aspects_negative", "language"]].head(8)

In [ ]:
# Câu hỏi mở đầu: khách CHÊ điều gì nhiều nhất?
complaint_counts = (full.explode("aspects_negative")["aspects_negative"]
                    .value_counts())
print(complaint_counts)

fig, ax = plt.subplots(figsize=(6, 3))
complaint_counts.sort_values().plot.barh(ax=ax, color="#1E93AB")
ax.set_title("Khía cạnh bị chê trong 24 review demo")
ax.set_xlabel("số review")
plt.tight_layout()
plt.show()

In [ ]:
# Và bảng cảm xúc × ngôn ngữ — thứ không thể có nếu không trích xuất
pd.crosstab(full["language"], full["sentiment"])

## 5. Đánh giá: LLM có thật sự hơn baseline không?

### 5a. Bộ nhãn gán tay (gold labels)

Chúng tôi đã **tự đọc và gán nhãn** 23 review (bỏ review 24 — rỗng, sẽ dùng riêng ở mục hậu kiểm).
Trong bài tập lớn, nhóm phải tự làm việc này với **≥100 review** — có mô tả quy trình gán.

In [ ]:
GOLD = {
 "1": {
  "sentiment": "positive",
  "aspects": [
   "location",
   "host"
  ],
  "language": "en"
 },
 "2": {
  "sentiment": "mixed",
  "aspects": [
   "location",
   "cleanliness"
  ],
  "language": "en"
 },
 "3": {
  "sentiment": "positive",
  "aspects": [
   "cleanliness",
   "host"
  ],
  "language": "es"
 },
 "4": {
  "sentiment": "negative",
  "aspects": [
   "noise"
  ],
  "language": "es"
 },
 "5": {
  "sentiment": "mixed",
  "aspects": [
   "location",
   "amenities"
  ],
  "language": "pt"
 },
 "6": {
  "sentiment": "mixed",
  "aspects": [
   "value"
  ],
  "language": "en"
 },
 "7": {
  "sentiment": "mixed",
  "aspects": [
   "noise"
  ],
  "language": "fr"
 },
 "8": {
  "sentiment": "positive",
  "aspects": [
   "value",
   "cleanliness",
   "noise"
  ],
  "language": "en"
 },
 "9": {
  "sentiment": "positive",
  "aspects": [],
  "language": "es"
 },
 "10": {
  "sentiment": "negative",
  "aspects": [
   "cleanliness"
  ],
  "language": "en"
 },
 "11": {
  "sentiment": "negative",
  "aspects": [
   "host"
  ],
  "language": "en"
 },
 "12": {
  "sentiment": "positive",
  "aspects": [
   "cleanliness",
   "amenities",
   "host"
  ],
  "language": "pt"
 },
 "13": {
  "sentiment": "mixed",
  "aspects": [
   "location"
  ],
  "language": "en"
 },
 "14": {
  "sentiment": "positive",
  "aspects": [],
  "language": "en"
 },
 "15": {
  "sentiment": "mixed",
  "aspects": [],
  "language": "es"
 },
 "16": {
  "sentiment": "positive",
  "aspects": [
   "location",
   "cleanliness"
  ],
  "language": "it"
 },
 "17": {
  "sentiment": "positive",
  "aspects": [
   "noise",
   "amenities",
   "host"
  ],
  "language": "en"
 },
 "18": {
  "sentiment": "negative",
  "aspects": [
   "cleanliness"
  ],
  "language": "en"
 },
 "19": {
  "sentiment": "negative",
  "aspects": [
   "noise"
  ],
  "language": "en"
 },
 "20": {
  "sentiment": "mixed",
  "aspects": [
   "location",
   "value"
  ],
  "language": "es"
 },
 "21": {
  "sentiment": "negative",
  "aspects": [
   "host"
  ],
  "language": "en"
 },
 "22": {
  "sentiment": "positive",
  "aspects": [
   "amenities",
   "location"
  ],
  "language": "en"
 },
 "23": {
  "sentiment": "positive",
  "aspects": [],
  "language": "und"
 }
}
GOLD = {int(k): v for k, v in GOLD.items()} if isinstance(list(GOLD)[0], str) else GOLD
print(f"Bộ đánh giá: {len(GOLD)} review gán tay")

In [ ]:
# 5b. Chấm điểm: sentiment & language = accuracy; aspects = micro-F1 trên tập nhãn
def score_flat(pred: dict[int, str], field: str) -> float:
    hit = sum(pred[rid] == g[field] for rid, g in GOLD.items())
    return hit / len(GOLD)

def score_aspects(pred: dict[int, set]) -> float:
    tp = fp = fn = 0
    for rid, g in GOLD.items():
        p, t = set(pred[rid]), set(g["aspects"])
        tp += len(p & t); fp += len(p - t); fn += len(t - p)
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0

llm_sent  = {rid: full.loc[rid, "sentiment"] for rid in GOLD}
llm_lang  = {rid: full.loc[rid, "language"] for rid in GOLD}
llm_asp   = {rid: set(full.loc[rid, "aspects_positive"]) | set(full.loc[rid, "aspects_negative"])
             for rid in GOLD}
base_sent = {rid: df.loc[rid, "base_sent"] for rid in GOLD}
base_lang = {rid: df.loc[rid, "base_lang"] for rid in GOLD}
base_asp  = {rid: set(df.loc[rid, "base_aspects"]) for rid in GOLD}

scoreboard = pd.DataFrame({
    "Baseline": [score_flat(base_sent, "sentiment"), score_aspects(base_asp), score_flat(base_lang, "language")],
    "LLM":      [score_flat(llm_sent, "sentiment"),  score_aspects(llm_asp),  score_flat(llm_lang, "language")],
}, index=["Cảm xúc (accuracy)", "Khía cạnh (micro-F1)", "Ngôn ngữ (accuracy)"]).round(2)
scoreboard

LLM hơn hẳn — **nhưng không hoàn hảo**. Con số chưa đủ: phải nhìn vào *chỗ sai*.

In [ ]:
# 5c. Phân tích lỗi: LLM sai ở những review nào?
errors = []
for rid, g in GOLD.items():
    diff = []
    if llm_sent[rid] != g["sentiment"]:
        diff.append(f"sentiment: {llm_sent[rid]} (đúng: {g['sentiment']})")
    if llm_lang[rid] != g["language"]:
        diff.append(f"language: {llm_lang[rid]} (đúng: {g['language']})")
    extra = llm_asp[rid] - set(g["aspects"])
    if extra:
        diff.append(f"khía cạnh bịa thêm: {sorted(extra)}")
    if diff:
        errors.append({"review_id": rid, "comments": df.loc[rid, "comments"][:60],
                       "lỗi": "; ".join(diff)})
pd.DataFrame(errors)

Ba lỗi trên là ba **kiểu lỗi kinh điển** của LLM:

1. **Khắt khe/châm chước khác người gán** (review 2: `negative` thay vì `mixed`) — ranh giới nhãn
   phải được định nghĩa rõ trong hướng dẫn gán *và* trong prompt.
2. **Nhầm ngôn ngữ gần nhau** trên câu ngắn (review 9: es → pt).
3. **Bịa khía cạnh không có trong văn bản** (review 14) — hallucination thật sự, nguy hiểm nhất
   vì trôi qua mắt người đọc lướt.

Sửa prompt xong phải **đo lại trên đúng bộ gold** — như sửa code phải chạy lại test.

In [ ]:
# 5d. Hậu kiểm (post-check) tự động: bắt các kết quả "đáng ngờ"
suspicious = []
for rid, r in ok.items():
    text = df.loc[rid, "comments"]
    n_aspects = len(r.aspects_positive) + len(r.aspects_negative)
    if len(text.strip()) == 0 and (n_aspects > 0 or r.sentiment != "mixed"):
        suspicious.append((rid, "review rỗng mà vẫn có nhãn!"))
    if len(text.split()) <= 4 and n_aspects >= 2:
        suspicious.append((rid, f"review {len(text.split())} từ mà ra {n_aspects} khía cạnh"))
for rid, why in suspicious:
    print(f"⚠️ review {rid}: {why} — {df.loc[rid, 'comments']!r}")

Review 24 (rỗng) bị bắt ngay: prompt đã dặn *"review không có nội dung thì trả mixed + danh sách rỗng"*
mà model vẫn trả `positive`. **Schema chặn được nhãn lạ, nhưng không chặn được nhãn "hợp lệ mà vô
căn cứ"** — vì thế cần thêm tầng hậu kiểm bằng quy tắc (tư duy QA buổi 10 áp cho output LLM).

In [ ]:
# 5e. Chi phí: cộng token đã dùng và quy ra tiền nếu chạy quy mô thật
tin = sum(v["in_tok"] for v in LLM_CACHE.values())
tout = sum(v["out_tok"] for v in LLM_CACHE.values())
print(f"24 review demo: {tin:,} token vào + {tout:,} token ra")

# Giá gemini-3.1-flash-lite (07/2026): $0.25 / 1M token vào, $1.50 / 1M token ra
# Free tier flash-lite thời điểm soạn bài: ~1.000 request/ngày (kiểm tra lại trong AI Studio)
per_review_in, per_review_out = tin / 24, tout / 24
for n in [2_000, 100_000, 690_000]:
    cost = (n * per_review_in * 0.25 + n * per_review_out * 1.50) / 1e6
    days_free = n / 1_000
    print(f"{n:>9,} review  ≈  ${cost:>6,.2f} trả phí   |   ~{days_free:,.0f} ngày nếu chạy free tier")

Kết luận thực dụng: **chọn mẫu thông minh** (vài nghìn review đúng câu hỏi) + **cache** thì free tier
đủ cho bài tập lớn; chạy "cả thành phố" là quyết định có giá tiền cụ thể — và giờ bạn tính được nó.

## 6. Bài tập tại lớp

Làm ngay tại chỗ, 15–20 phút. Sửa trực tiếp các cell dưới.

### Bài 1 — Mở rộng danh mục khía cạnh

Review 15 nhắc đến **an toàn** (*"el barrio se siente inseguro"*) nhưng enum của ta không có nhãn
`safety` nên thông tin này bị rơi mất. Hãy:

1. Thêm `"safety"` vào `Aspect` và thêm từ khoá cho baseline (`ASPECT_KEYWORDS`).
2. Chạy lại trích xuất cho **riêng review 15** (xoá cache của nó trước: `LLM_CACHE.pop("15", None)`)
   — nếu không có API key, hãy *tự đóng vai LLM*: viết tay JSON kết quả đúng schema mới và nhét vào
   cache, rồi chạy lại phần đánh giá.
3. Nhãn gold của review 15 cũng phải sửa theo. Điểm số thay đổi thế nào?

In [ ]:
# TODO Bài 1 — sửa và chạy lại (scaffold sẵn để cell luôn chạy được):
Aspect2 = Literal["location", "cleanliness", "host", "value", "noise", "amenities", "safety"]

class ReviewInfo2(ReviewInfo):
    aspects_positive: list[Aspect2]
    aspects_negative: list[Aspect2]

ASPECT_KEYWORDS["safety"] = r"inseguro|unsafe|safety|seguro de noche"   # TODO: mở rộng thêm
print("Schema mới có nhãn safety:", "safety" in str(ReviewInfo2.model_json_schema()))

### Bài 2 — Viết thêm một quy tắc hậu kiểm

Mục 5d mới có 2 quy tắc. Hãy thêm quy tắc thứ ba: **khía cạnh được gán phải "có dấu vết" trong văn
bản** — với mỗi aspect mà LLM trả về, nếu văn bản *không khớp* từ khoá nào của aspect đó
(dùng `ASPECT_KEYWORDS`), in cảnh báo. Quy tắc này có bắt được lỗi của review 14 không? Nó có thể
báo *oan* trong trường hợp nào? (gợi ý: review viết bằng ngôn ngữ mà từ khoá chưa phủ)

In [ ]:
# TODO Bài 2 — hoàn thiện hàm dưới:
def check_aspect_evidence(rid: int, r: ReviewInfo) -> list[str]:
    text = df.loc[rid, "comments"]
    warnings = []
    for aspect in set(r.aspects_positive) | set(r.aspects_negative):
        pat = ASPECT_KEYWORDS.get(aspect, "")
        if pat and not re.search(pat, text, flags=re.I):
            warnings.append(f"aspect '{aspect}' không có dấu vết trong văn bản")
    return warnings

for rid, r in ok.items():
    for w in check_aspect_evidence(rid, r):
        print(f"⚠️ review {rid}: {w} — {df.loc[rid, 'comments'][:50]!r}")

### Bài 3 — Sửa prompt, đo lại

Lỗi review 2 (`negative` thay vì `mixed`) đến từ ranh giới nhãn mờ. Sửa `PROMPT_TEMPLATE`: thêm định
nghĩa *"mixed = có cả ý khen lẫn ý chê"*, rồi (nếu có key) xoá cache review 2 và chạy lại. Điểm
sentiment tăng không? **Lưu ý:** mỗi lần sửa prompt là một "phiên bản" — ghi lại prompt nào cho điểm
nào, đừng sửa mò.

In [ ]:
# TODO Bài 3 — viết prompt v2 của bạn:
PROMPT_V2 = PROMPT_TEMPLATE.replace(
    "điền đúng schema.",
    "điền đúng schema. Quy ước: 'mixed' = có cả ý khen lẫn ý chê trong cùng review.",
)
print(PROMPT_V2)

## 7. Thử thách về nhà 🏆

Lấy **200 review thật** của một thành phố trong đề bài tập lớn và chạy toàn bộ pipeline của buổi này:

1. Tải `reviews.csv.gz` (URL mẫu trong cell dưới), lấy 200 review mới nhất có độ dài ≥ 30 ký tự.
2. Chạy baseline + LLM (chú ý quota: ~200 request, nhớ `time.sleep` + cache ra file JSON).
3. Tự gán tay 20 review làm gold, tính scoreboard như mục 5.
4. Viết 5 câu nhận xét: LLM đáng dùng cho bảng `reviews` của thành phố này không? Chi phí ước tính
   nếu chạy đủ 100% review?

Nộp: notebook + file cache JSON (để người chấm chạy lại **không cần key** — như bài tập lớn).

In [ ]:
RUN_CHALLENGE = False   # đổi thành True khi làm ở nhà

if RUN_CHALLENGE:
    URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
           "2026-06-29/data/reviews.csv.gz")          # đổi theo thành phố nhóm bạn
    rv = pd.read_csv(URL)
    rv = rv[rv["comments"].str.len() >= 30]
    sample = rv.sort_values("date").tail(200)
    print(sample.shape)

---

## Tóm tắt buổi học

| Ý chốt | Vì sao quan trọng |
|---|---|
| Phi cấu trúc → **đổi dạng thành bảng** | Dùng lại toàn bộ toolbox pandas đã học |
| **Baseline trước, LLM sau** | Không có thước đo thì không biết LLM đáng giá không |
| Schema + enum + rate limit + **cache** | Pipeline LLM chạy lại được, không cháy quota |
| **Bộ nhãn tay** + hậu kiểm + đếm token | Chất lượng và chi phí đều phải *đo*, không *đoán* |

📌 Đây chính là kỹ năng lõi của **hợp phần LLM trong bài tập lớn** — xem đề để biết yêu cầu đầy đủ
(baseline, ≥100 nhãn tay, báo cáo chi phí, `--skip-llm`).